In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('merged_movies.csv')
df.head()

,id,park,address,closedcaptioning,moviename,rating,date,zip
0,1,Beverly Park,"{""latitude"": ""41.70822525"", ""longitude"": ""-87....",Yes,Fireproof,PG,2014-07-16T20:30:00.000,60655.0
1,2,Holstein Park,"{""latitude"": ""41.9219017"", ""longitude"": ""-87.6...",Yes,The Goonies,PG,2014-08-28T20:00:00.000,60647.0
2,3,Dvorak Park,"{""latitude"": ""41.85511398"", ""longitude"": ""-87....",Yes,Despicable Me 2,PG,2014-06-20T21:00:00.000,60608.0
3,4,White (Willye B.) Park,"{""latitude"": ""42.01991653"", ""longitude"": ""-87....",Yes,Cloudy With a Chance of Meatballs 2,PG,2014-08-27T20:00:00.000,60626.0
4,5,Stanton Park,"{""latitude"": ""41.90510559"", ""longitude"": ""-87....",Yes,Despicable Me 2,PG,2014-08-15T20:00:00.000,60610.0


In [2]:
import json

def extract_coords(val):
    try:
        data = json.loads(val.replace("'", '"')) 
        return f"{data['latitude']}, {data['longitude']}"
    except:
        return np.nan

df['coords'] = df['address'].apply(extract_coords)
df.head()

,id,park,address,closedcaptioning,moviename,rating,date,zip,coords
0,1,Beverly Park,"{""latitude"": ""41.70822525"", ""longitude"": ""-87....",Yes,Fireproof,PG,2014-07-16T20:30:00.000,60655.0,"41.70822525, -87.68319702"
1,2,Holstein Park,"{""latitude"": ""41.9219017"", ""longitude"": ""-87.6...",Yes,The Goonies,PG,2014-08-28T20:00:00.000,60647.0,"41.9219017, -87.68556213"
2,3,Dvorak Park,"{""latitude"": ""41.85511398"", ""longitude"": ""-87....",Yes,Despicable Me 2,PG,2014-06-20T21:00:00.000,60608.0,"41.85511398, -87.65414429"
3,4,White (Willye B.) Park,"{""latitude"": ""42.01991653"", ""longitude"": ""-87....",Yes,Cloudy With a Chance of Meatballs 2,PG,2014-08-27T20:00:00.000,60626.0,"42.01991653, -87.67008972"
4,5,Stanton Park,"{""latitude"": ""41.90510559"", ""longitude"": ""-87....",Yes,Despicable Me 2,PG,2014-08-15T20:00:00.000,60610.0,"41.90510559, -87.64409637"


In [3]:
%pip install geopy
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="daen323_project")

def fill_missing_zip(row):
    if pd.isna(row['zip']):
        try:
            location = geolocator.reverse(row['coords'])
            address = location.raw.get('address', {})
            return address.get('postcode', row['zip'])
        except:
            return row['zip']
    return row['zip']


df['zip'] = df.apply(fill_missing_zip, axis=1)
df['closedcaptioning'] = df['closedcaptioning'].map({'Yes': 'Y', 'No': 'N'}).fillna(df['closedcaptioning'])
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df.loc[df['date'].isna(), 'date'] = pd.Timestamp('2016-01-01')
df['zip'] = df['zip'].astype(str).str.replace('.0', '', regex=False)

df.head()

Note: you may need to restart the kernel to use updated packages.


,id,park,address,closedcaptioning,moviename,rating,date,zip,coords
0,1,Beverly Park,"{""latitude"": ""41.70822525"", ""longitude"": ""-87....",Y,Fireproof,PG,2014-07-16 20:30:00,60655,"41.70822525, -87.68319702"
1,2,Holstein Park,"{""latitude"": ""41.9219017"", ""longitude"": ""-87.6...",Y,The Goonies,PG,2014-08-28 20:00:00,60647,"41.9219017, -87.68556213"
2,3,Dvorak Park,"{""latitude"": ""41.85511398"", ""longitude"": ""-87....",Y,Despicable Me 2,PG,2014-06-20 21:00:00,60608,"41.85511398, -87.65414429"
3,4,White (Willye B.) Park,"{""latitude"": ""42.01991653"", ""longitude"": ""-87....",Y,Cloudy With a Chance of Meatballs 2,PG,2014-08-27 20:00:00,60626,"42.01991653, -87.67008972"
4,5,Stanton Park,"{""latitude"": ""41.90510559"", ""longitude"": ""-87....",Y,Despicable Me 2,PG,2014-08-15 20:00:00,60610,"41.90510559, -87.64409637"


In [4]:
df['date'] = pd.to_datetime(df['date']).dt.date
display(df.head())
df.to_csv('cleaned_merged_movies.csv', index=False)
print("Transformation complete! The cleaned file is ready in your DAEN323_groupproj folder.")

,id,park,address,closedcaptioning,moviename,rating,date,zip,coords
0,1,Beverly Park,"{""latitude"": ""41.70822525"", ""longitude"": ""-87....",Y,Fireproof,PG,2014-07-16,60655,"41.70822525, -87.68319702"
1,2,Holstein Park,"{""latitude"": ""41.9219017"", ""longitude"": ""-87.6...",Y,The Goonies,PG,2014-08-28,60647,"41.9219017, -87.68556213"
2,3,Dvorak Park,"{""latitude"": ""41.85511398"", ""longitude"": ""-87....",Y,Despicable Me 2,PG,2014-06-20,60608,"41.85511398, -87.65414429"
3,4,White (Willye B.) Park,"{""latitude"": ""42.01991653"", ""longitude"": ""-87....",Y,Cloudy With a Chance of Meatballs 2,PG,2014-08-27,60626,"42.01991653, -87.67008972"
4,5,Stanton Park,"{""latitude"": ""41.90510559"", ""longitude"": ""-87....",Y,Despicable Me 2,PG,2014-08-15,60610,"41.90510559, -87.64409637"


Transformation complete! The cleaned file is ready in your DAEN323_groupproj folder.


In [5]:
missing_after_fix = df[df['date'].isna()]
print(f"Remaining rows with missing dates: {len(missing_after_fix)}")

Remaining rows with missing dates: 0


In [ ]:
df = df.rename(columns={
    'moviename': 'Movie Name',
    'address': 'Address',
    'rating': 'Rating',
    'date': 'Date',
    'closedcaptioning': 'Closed Captioning',
    'zip': 'Zip',
    'park': 'Park Name'
})

df['Date'] = pd.to_datetime(df['Date'])
df['Zip'] = pd.to_numeric(df['Zip'], errors='coerce').fillna(0).astype(int)

final_columns = ['Movie Name', 'Address', 'Rating', 'Date', 'Closed Captioning', 'Zip', 'Park Name']
df_final = df[final_columns]

display(df_final.head())
print(df_final.dtypes)

df_final.to_csv('cleaned_merged_movies_final.csv', index=False)

,Movie Name,Address,Rating,Date,Closed Captioning,Zip,Park Name
0,Fireproof,"{""latitude"": ""41.70822525"", ""longitude"": ""-87....",PG,2014-07-16,Y,60655,Beverly Park
1,The Goonies,"{""latitude"": ""41.9219017"", ""longitude"": ""-87.6...",PG,2014-08-28,Y,60647,Holstein Park
2,Despicable Me 2,"{""latitude"": ""41.85511398"", ""longitude"": ""-87....",PG,2014-06-20,Y,60608,Dvorak Park
3,Cloudy With a Chance of Meatballs 2,"{""latitude"": ""42.01991653"", ""longitude"": ""-87....",PG,2014-08-27,Y,60626,White (Willye B.) Park
4,Despicable Me 2,"{""latitude"": ""41.90510559"", ""longitude"": ""-87....",PG,2014-08-15,Y,60610,Stanton Park


Movie Name                   object
Address                      object
Rating                       object
Date                 datetime64[ns]
Closed Captioning            object
Zip                           int64
Park Name                    object
dtype: object


In [ ]:
##################
# TODO Try and bypass geolocator api rate limit
###################

unique_parks = df[['Park Name', 'coords']].drop_duplicates(subset=['Park Name'])
park_address_map = {}

print(f"Starting lookup for {len(unique_parks)} unique parks...")

for index, row in unique_parks.iterrows():
    park_name = row['Park Name']
    coords = row['coords']
    
    if pd.isna(coords):
        if pd.isna(park_name): #If there is no park name on file pass the loop
            #park_address_map[park_name] = "Address Unknown"
            continue #Strictly for error handling

        else: #If not, impute missing address with geopy address. 
            location = geolocator.geocode(park_name)
            if location:
                park_address_map[park_name] = location.address
                print(f'Found address for: {park_name} based on park name.')
            else:
                print(f'Did not find address for {park_name} based on park name.')

        continue
        
    try:
        import time
        time.sleep(1) 
        location = geolocator.reverse(coords)
        park_address_map[park_name] = location.address
        print(f"Found address for: {park_name}")
    except:
        park_address_map[park_name] = "Address Not Found"

df_final['Address'] = df_final['Park Name'].map(park_address_map)

df_final.to_csv('cleaned_merged_movies_final.csv', index=False)
print("Finished! All 1,800 rows updated.")

Starting lookup for 284 unique parks...
Found address for: Beverly Park
Found address for: Holstein Park
Found address for: Dvorak Park
Found address for: White (Willye B.) Park
Found address for: Stanton Park
Found address for: Austin Town Hall Park
Found address for: Cragin Park
Found address for: Lawler Park
Found address for: Galewood Park
Found address for: Revere Park
Found address for: Piotrowski Park
Found address for: Grant Park
Found address for: West Lawn Park
Found address for: Oz Park
Found address for: Edgebrook Park
Found address for: Armour Square Park
Found address for: Horner Park
Found address for: Jonquil Playlot Park
Found address for: Mandrake Park
Found address for: Ward, A. Montgomery Park
Found address for: O'Hallaren Park
Found address for: Athletic Field Park
Found address for: Jefferson Memorial Park
Found address for: Wrightwood Park
Found address for: Ada Park
Found address for: Portage Park
Found address for: Midway Plaisance Park
Found address for: Frank

GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Grant+Park%3A+Grove+5&format=json&limit=1 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=1)"))

In [ ]:
initial_count = len(df_final)
df_final = df_final.dropna(subset=['Movie Name'])

df_final = df_final[df_final['Movie Name'].str.strip() != ""]

final_count = len(df_final)
print(f"Removed {initial_count - final_count} rows with blank movie names.")
print(f"Total rows remaining: {final_count}")

df_final.to_csv('cleaned_merged_movies_final.csv', index=False)

display(df_final.head())

In [ ]:

df_final['Year'] = pd.to_datetime(df_final['Date']).dt.year
df_final.loc[df_final['Year'] == 2016, 'Date'] = np.nan
final_columns = ['Movie Name', 'Address', 'Rating', 'Year', 'Date', 'Closed Captioning', 'Zip', 'Park Name']
df_final = df_final[final_columns]
df_final.to_csv('cleaned_merged_movies_final.csv', index=False)
print("Yearly counts (including 2016):")
print(df_final['Year'].value_counts().sort_index())
print("\nCheck 2016 dates (should all be NaT/NaN):")
display(df_final[df_final['Year'] == 2016].head())

In [ ]:
df_final['Rating'] = df_final['Rating'].fillna('NR')

In [ ]:
df_final['Closed Captioning'] = df_final['Closed Captioning'].fillna('N')

print(f"Missing CC values: {df_final['Closed Captioning'].isna().sum()}")

df_final.to_csv('cleaned_merged_movies_final.csv', index=False)
display(df_final.head())

In [ ]:
print("DATA TYPE CHECK")
print(df_final.dtypes)

print("\nMISSING DATA CHECK")
print(df_final.isna().sum())

print("\nUNIQUE YEARS CHECK")
print(df_final['Year'].unique())

print("\nSAMPLE OUTPUT")
display(df_final.head(10))